In [ ]:
!nvidia-smi


In [ ]:
!pip install vllm


In [ ]:
!vllm --help

In [ ]:
from huggingface_hub import snapshot_download

local_dir = "gemma_3_1b_it"

snapshot_download(
    repo_id="google/gemma-3-1b-it",
    local_dir=local_dir,
    local_dir_use_symlinks=False
)


In [ ]:
!ls -lh gemma_3_1b_it


In [ ]:
!vllm serve --help

In [ ]:
!vllm serve /content/gemma_3_1b_it \
    --port 8000 \
    --gpu-memory-utilization 0.95 \
    --max-model-len 4096 \
    --max-num-seqs 50 \
    --dtype auto \
    --tensor-parallel-size 1


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir="./qwen3",
    local_dir_use_symlinks=False
)


In [ ]:
!pip install openai


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

stream = client.chat.completions.create(
    model="/content/qwen3",
    messages=[
        {"role": "user", "content": "Write a short story about AI and GPUs."}
    ],
    max_tokens=256,
    temperature=0.7,
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)


In [ ]:
import asyncio
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

async def user_request(user_id: int):
    stream = client.chat.completions.create(
        model="/content/qwen3",
        messages=[
            {
                "role": "user",
                "content": f"User {user_id}: Write a short story about AI and GPUs."
            }
        ],
        max_tokens=256,
        temperature=0.7,
        stream=True
    )

    output = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta and delta.content:
            output += delta.content

    print(f"\n--- User {user_id} finished ({len(output)} chars) ---")

async def main():
    tasks = [user_request(i) for i in range(50)]
    await asyncio.gather(*tasks)

await main()


**Chunked prefill** (sometimes shown as `enable_chunked_prefill=True` in vLLM logs) is an **optimization for the *prompt processing* phase** of inference. It mainly helps when you have **long prompts and many concurrent users**.

---

## 1️⃣ What is “prefill” (baseline)

Inference has two phases:

### 🔹 Prefill phase

* The model **reads the prompt** (all input tokens)
* Builds the **KV cache** for those tokens
* This is **expensive** and happens *before* generation

### 🔹 Decode phase

* The model **generates new tokens**
* Much cheaper per token (KV cache already exists)

For a long prompt (e.g., 4k tokens), **prefill is the slowest part**.

---

## 2️⃣ The problem without chunked prefill

Without chunked prefill, vLLM does this:

```
User A: prefill 4000 tokens (blocking)
User B: waits
User C: waits
User D: waits
```

Problems:

* One long prompt **blocks the GPU**
* Other users experience **high latency**
* Bad tail latency (P95 / P99)

---

## 3️⃣ What chunked prefill does ✅

**Chunked prefill splits the prompt into smaller chunks** and processes them incrementally.

Instead of:

```
Prefill: 4000 tokens in one shot
```

vLLM does:

```
Prefill chunk 1: 512 tokens
Prefill chunk 2: 512 tokens
Prefill chunk 3: 512 tokens
...
```

And **interleaves chunks from different users**.

### Result:

```
A chunk → B chunk → C chunk → A chunk → D chunk → ...
```

---

## 4️⃣ Why chunked prefill is powerful

### ✅ Benefits

| Benefit           | Explanation                     |
| ----------------- | ------------------------------- |
| Lower latency     | Long prompts don’t block others |
| Fair scheduling   | All users make progress         |
| Better throughput | GPU always busy                 |
| Better UX         | First tokens appear sooner      |
| Safer memory      | Less sudden KV allocation       |

This is **critical** for:

* 50+ concurrent users
* Chat apps
* RAG (long documents)
* Streaming APIs

---

## 5️⃣ Chunked prefill vs KV cache (important distinction)

| Feature             | Role                                         |
| ------------------- | -------------------------------------------- |
| **KV cache**        | Stores past tokens so they aren’t recomputed |
| **Chunked prefill** | Controls *how* KV cache is built             |

👉 Chunked prefill **does NOT disable KV cache**
👉 It just **builds it gradually**

They work **together**.

---

## 6️⃣ When chunked prefill helps the most

### ✅ Very helpful

* Long prompts (2k–8k tokens)
* Many concurrent users
* Streaming responses
* RAG pipelines

### ❌ Less impact

* Very short prompts (<200 tokens)
* Single-user inference
* Offline batch jobs

---

## 7️⃣ Does chunked prefill reduce quality?

❌ No
❌ No change in output
❌ No change in probabilities

It only affects **scheduling**, not model math.

---

## 8️⃣ How to control it in vLLM

In recent vLLM versions, it is **enabled by default**.

You’ll see:

```
enable_chunked_prefill=True
```

You usually **should NOT disable it**.

If you did want to disable (not recommended):

```bash
--disable-chunked-prefill
```

---

## 9️⃣ How this helps *your* setup (T4 + 50 users)

For you:

* GPU: **T4**
* Users: **~50**
* Context: **4k**

Chunked prefill:

* Prevents one user from freezing the GPU
* Keeps latency acceptable
* Makes streaming feel responsive
* Increases stable concurrency

This is **one of the reasons vLLM can handle 50 users on one GPU**.

---

## 🔑 Mental model (easy to remember)

> **KV cache = memory**
> **Chunked prefill = how memory is filled**

---

## 🟢 TL;DR

* **Prefill** = reading prompt & building KV cache
* **Chunked prefill** = split prompt into smaller pieces
* Prevents long prompts from blocking others
* Improves latency, fairness, and throughput
* Safe, default, and recommended
* Essential for multi-user serving

